# YOLO26s training on LS-SSD dataset

This notebook converts the **LS-SSD** ship dataset (VOC-style) into the
format expected by the Ultralytics YOLOv8/YOLO26 training code and then
launches a training run using the `yolo26s` model.  The original data lives
in `ls-ssdd/` and contains Pascal‑VOC XML annotations and separate
`JPEGImages_sub_train`/`JPEGImages_sub_test` folders.

In [1]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil


def voc_to_yolo(root_dir, output_dir, splits=("train", "val", "test")):
    """Convert the LS-SSD VOC-style dataset to YOLO format.

    The function will:
    1. read the `train.txt`, `val.txt` and `test.txt` lists from ``root_dir``
    2. copy the corresponding JPEGs into ``output_dir`` under
       ``images/<split>``
    3. parse each XML file and write a YOLO-format label file ``<id>.txt`` in
       ``labels/<split>`` (class 0 = ship)
    4. emit a dataset YAML file that points to the new directory.

    Returns the path to the generated YAML file.
    """

    root_dir = Path(root_dir)
    ann_dir = root_dir / "Annotations_sub" / "Annotations_sub"
    img_train_dir = root_dir / "JPEGImages_sub_train" / "JPEGImages_sub_train"
    img_test_dir = root_dir / "JPEGImages_sub_test" / "JPEGImages_sub_test"

    out = Path(output_dir)
    for split in splits:
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

        list_file = root_dir / f"{split}.txt"
        if not list_file.exists():
            continue
        with open(list_file) as f:
            ids = [line.strip() for line in f if line.strip()]

        for idx in ids:
            # choose appropriate image directory
            if split in ("train", "val"):
                src_img_dir = img_train_dir
            else:
                src_img_dir = img_test_dir

            src_img = src_img_dir / f"{idx}.jpg"
            if src_img.exists():
                dst_img = out / "images" / split / src_img.name
                shutil.copy2(src_img, dst_img)

            xml_file = ann_dir / f"{idx}.xml"
            labels = []
            if xml_file.exists():
                tree = ET.parse(xml_file)
                root = tree.getroot()
                size = root.find("size")
                w = float(size.find("width").text)
                h = float(size.find("height").text)
                for obj in root.findall("object"):
                    cls = obj.find("name").text
                    if cls.lower() != "ship":
                        continue
                    bnd = obj.find("bndbox")
                    xmin = float(bnd.find("xmin").text)
                    ymin = float(bnd.find("ymin").text)
                    xmax = float(bnd.find("xmax").text)
                    ymax = float(bnd.find("ymax").text)
                    xmid = ((xmin + xmax) / 2) / w
                    ymid = ((ymin + ymax) / 2) / h
                    bw = (xmax - xmin) / w
                    bh = (ymax - ymin) / h
                    labels.append(f"0 {xmid:.6f} {ymid:.6f} {bw:.6f} {bh:.6f}")
            # write label file (empty when no objects present)
            with open(out / "labels" / split / f"{idx}.txt", "w") as lf:
                lf.write("\n".join(labels))

    # write dataset YAML
    yaml_path = out / "ls_ssdd.yaml"
    yaml_content = f"""path: {out.absolute()}
train: images/train
val: images/val

nc: 1
names:
  0: ship
"""
    yaml_path.write_text(yaml_content)
    return str(yaml_path)


# helper for training the tiny model; mostly copied from earlier notebook
from ultralytics import YOLO

def train_yolo26s(
    yaml_path: str,
    output_dir: str = "runs/ls_ssdd",
    epochs: int = 100,
    imgsz: int = 800,
    batch: int = 16,
    device: str = "0",
):
    """Train YOLO26s on the given dataset YAML."""
    model = YOLO("yolo26s.pt")
    model.train(
        data=yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        project=output_dir,
        name="yolo26s",
        plots=True,
        save=True,
        save_period=5,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.2,
        fliplr=0.5,
        flipud=0.5,
        mosaic=0.5,
        optimizer="MuSGD",
        lr0=0.01,
    )
    print(f"Training finished, best weights in {output_dir}/yolo26s/weights/best.pt")


In [2]:
# prepare the data and launch training

ls_root = Path("ls-ssdd")  # relative to workspace root
out_dir = Path("ls-ssdd/yolo")

print("converting dataset...")
yaml_path = voc_to_yolo(ls_root, out_dir)
print("dataset yaml written to", yaml_path)

converting dataset...
dataset yaml written to ls-ssdd\yolo\ls_ssdd.yaml


In [ ]:
# now train the small YOLO26s model
train_yolo26s(yaml_path, output_dir="./logs/ls_ssdd", epochs=24, imgsz=800, batch=8, device='0')

New https://pypi.org/project/ultralytics/8.4.19 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ls-ssdd\yolo\ls_ssdd.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=24, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolo26s


---

Once the run completes you'll find TensorBoard logs in `./logs/ls_ssdd/yolo26s`.
View with `tensorboard --logdir=./logs/ls_ssdd` or export the best weights with
`YOLO(...).export(...)` if you need ONNX/ONNX-TS/TorchScript formats.